# FÖ3 - förskjutningmetod för balkar

Innehåll:
1. Interaktivt exempel
2. Konsolbalk med punktlast
3. Konsolbalk med fördelad last
4. Substitution med numeriska värden

Notera: Stor del av koden i den här notebooken är tänkt för självstudier.


## 1. Interaktivt exampel som visar balknedböjningen $w(x)$ som funktion av de fyra frihetsgraderna $u_1-u_4$ 
Utböjningen för en balk med konstant EI belastad av enbart laster i ändarna får följande utböjningsfunktion $w(x)$ 
$$
    w(x) = N_1 u_1 + N_2 u_2 + N_3 u_3 + N_4 u_4
$$ 

Koden som visualiserar är inte relevant just nu utan studera resultatet med sliders.
* $u_1$,  $u_3$ är utböjningar [m]
* $u_2$,  $u_4$ är rotationer [rad] (positiva moturs) 

**Att göra**
Ändra värdena på de olika frihetsgraderna, ändra en i taget för att få en känsla för vad respektive fihetsgrad ($u_1-u_4$) representerar. Syftet är att du ska vara förstå vad exempelvis $u_3$ är, vad innebär det att den är låst till noll eller fri att ändras?


In [ ]:
import numpy as np
import plotly.graph_objects as go
from ipywidgets import interact, FloatSlider

L = 1.0  # fixed length

def hermite_shape_functions_unit(xi):
    """
    Hermite cubic (Euler-Bernoulli) shape functions for a unit-length element.
    Returns N1..N4 such that: w(xi) = N1*u1 + N2*u2 + N3*u3 + N4*u4
    """
    N1 = 1 - 3*xi**2 + 2*xi**3
    N2 = xi - 2*xi**2 + xi**3          # since L=1, the usual L factor is 1
    N3 = 3*xi**2 - 2*xi**3
    N4 = -xi**2 + xi**3                # since L=1
    return N1, N2, N3, N4

def beam_deflection_curve(u1, u2, u3, u4, npts=300):
    xi = np.linspace(0.0, 1.0, npts)
    N1, N2, N3, N4 = hermite_shape_functions_unit(xi)
    w = N1*u1 + N2*u2 + N3*u3 + N4*u4
    x = xi  # L=1
    return x, w

def plot_beam(u1=0.0, u2=0.0, u3=0.0, u4=0.0):
    x, w = beam_deflection_curve(u1, u2, u3, u4, npts=400)

    fig = go.Figure()

    # Undeformed centerline
    fig.add_trace(go.Scatter(
        x=[0.0, 1.0], y=[0.0, 0.0], mode='lines', name='odeformerad',
        line=dict(color='gray', width=2, dash='dash')
    ))

    # Deformed shape (no scaling; y-axis fixed to [-1,1])
    fig.add_trace(go.Scatter(
        x=x, y=w, mode='lines', name='Deformerad',
        line=dict(color='#1f77b4', width=3)
    ))

    # Node markers with labels (x=0,1; y=u1,u3)
    fig.add_trace(go.Scatter(
        x=[0.0, 1.0], y=[u1, u3], mode='markers+text', name='Noder',
        marker=dict(color='crimson', size=10),
        text=[f"u₁={u1:.2f} u₂={u2:.2f}", f"u₃={u3:.2f} u₄={u4:.2f}"],
        textposition='top center', hoverinfo='skip'
    ))

    fig.update_layout(
        title="Transversell utböjning w(x) av en balk med alla frihetsgrader föreskrivna",
        xaxis_title="x",
        yaxis_title="w",
        template="plotly_white",
        showlegend=True,
        width=900, height=420,
        margin=dict(l=40, r=20, t=60, b=40)
    )

    # Fixed axes as requested
    fig.update_xaxes(range=[-.10, 1.1], zeroline=True, showgrid=True)
    fig.update_yaxes(range=[-1.2, 1.2], zeroline=True, showgrid=True)

    fig.show()

# Interactive sliders for u1..u4
_ = interact(
    plot_beam,
    u1=FloatSlider(value=.5, min=-1.0, max=1.0, step=0.1, description='u₁ (w@x=0)'),
    u2=FloatSlider(value=1.0, min=-2.0, max=2.0, step=0.1, description='u₂ (θ@x=0)'),
    u3=FloatSlider(value=-.2, min=-1.0, max=1.0, step=0.1, description='u₃ (w@x=1)'),
    u4=FloatSlider(value=1, min=-2.0, max=2.0, step=0.1, description='u₄ (θ@x=1)'),
)

interactive(children=(FloatSlider(value=0.5, description='u₁ (w@x=0)', max=1.0, min=-1.0), FloatSlider(value=1…

## 2. Konsolbalk som belastas av en nedåtriktad punktlast i sin högra ände

In [ ]:
from mtm026 import *

EI, L, P = sp.symbols("EI L P") #  definiera symboliska variabler
a3, a4, r1, r2 = sp.symbols("a3, a4, r1, r2") # obekanta

# Bestäm elementstyvhetsmatrisen med funktionen `Ke_balk`
K = Ke_balk(EI=EI, L=L) # samma sak som Ke_balk(EI, L) men med namngivna argument
displayvar("K", sp.Matrix(K))

<IPython.core.display.Math object>

0

### Randvillkor
* Definiera förskjutningsvektorn
* Definiera reaktionsvektorn som innehåller reaktionskrafter och reaktionsmoment

Använder funktionen `sp.Matrix` för att skapa vektorer och argumentet till funktionen är en lista på värden/variabler

In [3]:
a = sp.Matrix([0, 0, a3, a4]) # [0 0 a₃ a₄]ᵀ a - förskjutningsvektor    
r = sp.Matrix([r1, r2, 0, 0]) # [r₁ r₂ 0 0]ᵀ r - reaktionskrafter  
fy = sp.Matrix([0, 0, -P, 0]) # [0 0 -P 0]  fy - yttre lastvektor
displayvar("a", a)
displayvar("r", r)
displayvar("f", fy)

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

### Definiera ekvationssystem och lös detta
* Använd `sp.Eq(Vänsterled, högerled)` för att definiera ekvationssystemet
* Använd `sp.solve(ekvationer, obekanta)` för att lösa en uppsättning ekvationer för de angivna variablerna i listan `obekanta`

In [4]:
ekvsys = sp.Eq(K*a, r + fy)
display(ekvsys) # Skriv ut ekvationssystemet (men inte så överskådligt)
# display_eqnsys(K, a, r + fy) # MTM026-funktion för att skriva ut ekvationssystemet på ett trevligt format


Eq(Matrix([
[ 6*EI*a4/L**2 - 12*EI*a3/L**3],
[     2*EI*a4/L - 6*EI*a3/L**2],
[-6*EI*a4/L**2 + 12*EI*a3/L**3],
[     4*EI*a4/L - 6*EI*a3/L**2]]), Matrix([
[r1],
[r2],
[-P],
[ 0]]))

### Lös ekvationssystemet
Använd funktionen ```sp.solve(ekvationer, obekanta)```

In [5]:
obekanta = [a3, a4, r1, r2]
sol = sp.solve(ekvsys, obekanta) # sol = solution

displayvar("sol", sol) # en dictionary som innehåller lösningen (solution) till ekvationsys.

<IPython.core.display.Math object>

## 3. Konsolbalk belastad av fördelad last över hela längden 
Samma struktur som exemplet ovan men balken är istället belastad av en fördelad last med konstant intensitet q0.

Enda skillnaden i lösningen blir i lastvektorn eftersom strukturen är densamma (=> samma K) och upplagen är desamma (=> samma struktur på a och r)

In [6]:
q0 = sp.symbols("q0")

# yttre lastvektor
fy = fe_balk(q=-q0, L=L) # skriver över f_y från förra exemplet
displayvar("f_y", fy)
# fy = sp.Matrix([0, 0, -P, 0]) # i förra exemplet med bara en punktlast 

ekvsys = sp.Eq(K*a, r+fy)

sol = sp.solve(ekvsys, obekanta)
displayvar("sol", sol) 

# substituera in värden i lösningen
a = a.subs(sol)
r = r.subs(sol)
displayvar("a", a)
displayvar("r", r)

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

## 4. Numeriska värden
Ifall man vill konvertera sitt symboliska yttryck för att räkna ut ett numeriskt svar gör man det enklast med `subs` där man skickar in en `dictionary` som innehåller variabler och värden man vill ersätta med.

In [7]:
data = {
    L: 2,           # [m]
    EI: 200e9*1e-5, # [m^4] 
    q0: 1e3,        # [N/m] = 1kN/m
}

a_num = a.subs(data) # substituera alla variabler i uttrycket a med de värden som finns i variablen data
r_num = r.subs(data)

displayvar("a", a)
displayvar("a_{num}", a_num)

# ofta vill man svara med ett visst antal väsentliga siffror
displayvar("a_{num}", a_num.evalf(3)) # 3 väsentliga siffror
displayvar("r_{num}", r_num.evalf(3)) 
displayvar("r_{num}", r_num, accuracy=3) # displayvar tar ett extra argument 'accuracy' som gör samma sak 


<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>